# Contact-Predicted SAAP Analysis

Tests whether contact-predicted SAAP peptides (proximal to destabilizing missense mutations) are:
1. **Detected more often** in TMT channels whose patient carries the corresponding destabilizing missense
2. **More abundant** (higher RI intensity) in carrier channels vs non-carrier channels

Compares destabilizing (AM+SPURS, SPURS only, AM only) vs neutral contact predictions as a negative control.

**Logic:**
- Contact FASTA entries tagged `destab_contact` or `neutral_contact` in the description
- For each detected contact PSM in a plex:
  - Parse gene + contact position from the entry name
  - Find which plex patients have a destabilizing missense within 3Å of that position
  - Compare RI intensity: carrier channels vs non-carrier channels
  - Compute detection rate per channel group

In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

# ── CONFIG ────────────────────────────────────────────────────────────────────
REPO_DIR     = '/home/leduc.an/AAS_Evo_project/AAS_Evo'
TMT_MAP      = f'{REPO_DIR}/metadata/PDC_meta/pdc_file_tmt_map.tsv'
GDC_META     = f'{REPO_DIR}/metadata/GDC_meta/gdc_meta_matched.tsv'
MISSENSE     = '/scratch/leduc.an/AAS_Evo/ANALYSIS/all_missense_with_spurs.tsv'
REF_FASTA    = '/scratch/leduc.an/AAS_Evo/SEQ_FILES/uniprot_human_canonical.fasta'
RESULTS_BASE = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/results_contact'
CONTACT_DIR  = '/scratch/leduc.an/AAS_Evo/SPURS/contact_maps'
DDG_DIR      = '/scratch/leduc.an/AAS_Evo/SPURS/ddg_matrices'
PLEX_LIST    = '/scratch/leduc.an/AAS_Evo/MS_SEARCH/plex_list.txt'
OUT_DIR      = '/scratch/leduc.an/AAS_Evo/ANALYSIS/contact_saap'
os.makedirs(OUT_DIR, exist_ok=True)

AM_THRESHOLD    = 0.564
SPURS_THRESHOLD = 0.5
AM_BENIGN_MAX   = 0.1
GNOMAD_NEUTRAL  = 0.1
GNOMAD_MAX      = 0.01
VAF_THRESHOLD   = 0.3
DIST_THRESHOLD  = 3.0   # Å Cα-Cα (must match generate_contact_saap_fastas.py)

TMT_CHANNEL_MAP = {
    'tmt_126':'126','tmt_127n':'127N','tmt_127c':'127C',
    'tmt_128n':'128N','tmt_128c':'128C','tmt_129n':'129N',
    'tmt_129c':'129C','tmt_130n':'130N','tmt_130c':'130C',
    'tmt_131':'131N','tmt_131c':'131C','tmt_126c':'126C','tmt_134n':'134N',
}
CHANNEL_ORDER = ['126','127N','127C','128N','128C','129N','129C','130N','130C','131N','131C']


In [ ]:
# ── LOAD METADATA ─────────────────────────────────────────────────────────────
tmt = pd.read_csv(TMT_MAP, sep='\t')
gdc = pd.read_csv(GDC_META, sep='\t')
if 'file_id' in gdc.columns and 'gdc_file_id' not in gdc.columns:
    gdc = gdc.rename(columns={'file_id':'gdc_file_id'})

with open(PLEX_LIST) as f:
    plex_ids = [l.strip() for l in f if l.strip()]
# Only plexes with results_contact results
plex_ids = [p for p in plex_ids
            if os.path.exists(os.path.join(RESULTS_BASE, p, 'combined_protein.tsv'))]

uuid_to_stype = gdc.set_index('gdc_file_id')['sample_type'].to_dict()
case_sample_to_uuid = gdc.set_index(['case_submitter_id','sample_type'])['gdc_file_id'].to_dict()

print(f'Plexes with results_contact: {len(plex_ids)}')

In [ ]:
# ── LOAD & FILTER MISSENSE TABLE ──────────────────────────────────────────────
print('Loading missense table...')
miss = pd.read_csv(MISSENSE, sep='\t', low_memory=False,
                   usecols=['sample_id','SYMBOL','Protein_position',
                            'Amino_acids','VAF','gnomADe_AF',
                            'am_pathogenicity','spurs_ddg'])
miss['VAF']             = pd.to_numeric(miss['VAF'],             errors='coerce')
miss['gnomADe_AF']      = pd.to_numeric(miss['gnomADe_AF'],      errors='coerce').fillna(0)
miss['am_pathogenicity']= pd.to_numeric(miss['am_pathogenicity'],errors='coerce')
miss['spurs_ddg']       = pd.to_numeric(miss['spurs_ddg'],       errors='coerce')
miss['pos']             = pd.to_numeric(
    miss['Protein_position'].astype(str).str.split('-').str[0], errors='coerce')

base_ok   = (miss['VAF'] >= VAF_THRESHOLD) & (miss['gnomADe_AF'] < GNOMAD_MAX)
am_pos    = miss['am_pathogenicity'] >= AM_THRESHOLD
spurs_pos = miss['spurs_ddg']        >= SPURS_THRESHOLD

def classify_miss(am, spurs):
    a = pd.notna(am)    and am    >= AM_THRESHOLD
    s = pd.notna(spurs) and spurs >= SPURS_THRESHOLD
    if a and s:  return 'Both AM+SPURS'
    if s:        return 'SPURS only'
    if a:        return 'AM only'
    if pd.notna(am) and am <= AM_BENIGN_MAX: return 'Neutral'
    return None

destab  = miss[(am_pos | spurs_pos) & base_ok].copy()
neutral = miss[(miss['am_pathogenicity'] <= AM_BENIGN_MAX) &
               (miss['gnomADe_AF'] >= GNOMAD_NEUTRAL)].copy()

destab['group']  = destab.apply(lambda r: classify_miss(r['am_pathogenicity'], r['spurs_ddg']), axis=1)
neutral['group'] = 'Neutral'
all_miss = pd.concat([destab, neutral], ignore_index=True)
all_miss_indexed = all_miss.groupby('sample_id')

all_processed_uuids = set(miss['sample_id'].unique())
print(f'Destab: {len(destab):,} | Neutral: {len(neutral):,}')

In [ ]:
# ── CONTACT MAP HELPERS + SEQUENCE LOADING ────────────────────────────────────
import re as _re

gene_to_acc = {f.name.split('.')[1]: f.name.split('.')[0]
               for f in Path(DDG_DIR).glob('*.ddg_matrix.tsv')}

print('Loading reference sequences...')
seqs = {}
cur_acc = None
with open(REF_FASTA) as fh:
    for line in fh:
        line = line.rstrip()
        if line.startswith('>'):
            parts = line.split('|')
            cur_acc  = parts[1] if len(parts) >= 3 else line[1:].split()[0]
            m_gene   = _re.search(r'GN=(\S+)', line)
            cur_gene = m_gene.group(1) if m_gene else None
            seqs[cur_acc] = []
            if cur_gene and cur_gene not in gene_to_acc:
                gene_to_acc[cur_gene] = cur_acc
        elif cur_acc:
            seqs[cur_acc].append(line)
seqs = {acc: ''.join(s) for acc, s in seqs.items()}
print(f'  {len(seqs):,} sequences | {len(gene_to_acc):,} gene->acc mappings')

# ── Tryptic digestion ─────────────────────────────────────────────────────────
def _tryptic_cuts(seq):
    cuts = [-1]
    for i, aa in enumerate(seq):
        if aa in 'KR' and (i + 1 >= len(seq) or seq[i + 1] != 'P'):
            cuts.append(i)
    cuts.append(len(seq) - 1)
    return cuts

def get_wt_peptides(acc, contact_pos_1based):
    """Return WT tryptic peptide sequences (0- and 1-missed cleavage) covering contact_pos."""
    seq = seqs.get(acc, '')
    if not seq or contact_pos_1based < 1 or contact_pos_1based > len(seq):
        return []
    cuts = _tryptic_cuts(seq)
    idx = contact_pos_1based - 1
    results = set()
    for i in range(len(cuts) - 1):
        for j in range(i + 1, min(i + 3, len(cuts))):
            start = cuts[i] + 1
            end   = cuts[j] + 1
            if start <= idx < end:
                pep = seq[start:end]
                if len(pep) >= 6:
                    results.add(pep)
    return list(results)

# ── Contact map helpers ───────────────────────────────────────────────────────
cm_cache     = {}
nearby_cache = {}

def load_contact_map(acc):
    cdir = Path(CONTACT_DIR)
    for npy_path in sorted(cdir.glob(f'AF-{acc}-*F1.npy')):
        csv_path = npy_path.with_suffix('.csv')
        if not csv_path.exists(): continue
        try:
            meta = pd.read_csv(csv_path, index_col=0)
            dm   = np.load(npy_path)
            p2i  = {int(row['id']): i for i, row in meta.iterrows() if pd.notna(row['id'])}
            return p2i, dm
        except Exception:
            continue
    return None, None

def get_nearby(acc, pos):
    key = (acc, pos)
    if key not in nearby_cache:
        if acc not in cm_cache:
            cm_cache[acc] = load_contact_map(acc)
        p2i, dm = cm_cache[acc]
        if dm is None:
            nearby_cache[key] = []
        else:
            idx = p2i.get(pos)
            if idx is None:
                nearby_cache[key] = []
            else:
                pos_arr = np.array(list(p2i.keys()),   dtype=np.int32)
                idx_arr = np.array(list(p2i.values()), dtype=np.int32)
                d_vals  = dm[idx][idx_arr]
                mask    = (d_vals > 0) & (d_vals < DIST_THRESHOLD) & (pos_arr != pos)
                nearby_cache[key] = pos_arr[mask].tolist()
    return nearby_cache[key]

def find_ri_cols(df):
    found = {}
    for ch in CHANNEL_ORDER + ['126C','132N','132C','133N','134N']:
        if ch in df.columns:
            found[ch] = ch; continue
        cands = [c for c in df.columns if c.startswith('Intensity') and c.endswith(f'_{ch}')]
        if cands: found[ch] = cands[0]
    return found

def get_channel_uuid_map(plex_id):
    pt = tmt[tmt['run_metadata_id'] == plex_id][['tmt_channel','case_submitter_id','sample_type']].drop_duplicates()
    pt = pt[~pt['case_submitter_id'].str.lower().isin(['ref','reference','pooled','pool','nan'])]
    pt['channel'] = pt['tmt_channel'].map(TMT_CHANNEL_MAP)
    pt = pt.dropna(subset=['channel'])
    merged = pt.merge(gdc[['gdc_file_id','case_submitter_id','sample_type']],
                      on=['case_submitter_id','sample_type'], how='left')
    return merged.dropna(subset=['gdc_file_id']).drop_duplicates('channel').set_index('channel')['gdc_file_id'].to_dict()

# Build canonical tryptic peptide set for proteome-level false-positive check
print('Building canonical peptide set...')
canonical_peptides = set()
for _seq in seqs.values():
    _cuts = [-1]
    for _i, _aa in enumerate(_seq):
        if _aa in 'KR' and (_i + 1 >= len(_seq) or _seq[_i + 1] != 'P'):
            _cuts.append(_i)
    _cuts.append(len(_seq) - 1)
    for _i in range(len(_cuts) - 1):
        _pep = _seq[_cuts[_i] + 1 : _cuts[_i + 1] + 1]
        if len(_pep) >= 6:
            canonical_peptides.add(_pep)
print(f'  {len(canonical_peptides):,} canonical tryptic peptides indexed')

print('Helpers defined')


In [ ]:
# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
records = []
n_done = n_no_psm = n_no_wt = 0

for pi, plex_id in enumerate(plex_ids):
    if pi % 5 == 0:
        print(f'  {pi}/{len(plex_ids)} plexes, {len(records):,} records', flush=True)

    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files:
        psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, 'psm.tsv')))
    if not psm_files:
        n_no_psm += 1; continue

    psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    ri_col_map = find_ri_cols(psm)
    if not ri_col_map: continue

    ch_uuid = get_channel_uuid_map(plex_id)
    channels_with_genomics = {ch for ch, uuid in ch_uuid.items()
                              if uuid in all_processed_uuids}
    if len(channels_with_genomics) < 2: continue

    # Aggregate RI per peptide across all PSMs in the plex
    ri_cols_list = list(ri_col_map.values())
    pep_ri = (psm.groupby('Peptide')[ri_cols_list]
                 .sum()
                 .rename(columns={v: k for k, v in ri_col_map.items()}))

    # Get plex patient missense
    plex_uuids = set(ch_uuid.values())
    plex_miss_frames = [all_miss_indexed.get_group(uid)
                        for uid in plex_uuids if uid in all_miss_indexed.groups]
    plex_miss = pd.concat(plex_miss_frames) if plex_miss_frames else pd.DataFrame()
    if len(plex_miss) == 0: continue

    contact_mask = psm['Entry Name'].astype(str).str.endswith('-contact')
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    seen = set()
    for _, row in contact_psm.iterrows():
        entry_name = str(row.get('Entry Name', ''))
        protein_id  = str(row.get('Protein ID', ''))
        pep_seq     = str(row.get('Peptide', ''))

        dedup_key = (entry_name, protein_id, pep_seq)
        if dedup_key in seen: continue
        seen.add(dedup_key)

        m_gene = _re.match(r'^([A-Z0-9]+)-contact$', entry_name)
        if not m_gene: continue
        gene = m_gene.group(1)

        m_swap = _re.search(r'-([A-Z])(\d+)([A-Z])-[A-Z0-9]{4}$', protein_id)
        if not m_swap: continue
        wt_aa, contact_pos, alt_aa = m_swap.group(1), int(m_swap.group(2)), m_swap.group(3)

        acc = gene_to_acc.get(gene)
        if not acc: continue

        wt_peps = get_wt_peptides(acc, contact_pos)
        wt_pep  = next((p for p in wt_peps if p in pep_ri.index), None)
        if wt_pep is None:
            n_no_wt += 1; continue
        if pep_seq not in pep_ri.index: continue

        gene_miss = plex_miss[plex_miss['SYMBOL'] == gene].dropna(subset=['pos'])
        if len(gene_miss) == 0: continue

        # Build separate carrier sets per source group; also track which missense
        # positions drive each group (for downstream multi-site analysis).
        group_carriers  = defaultdict(set)   # source_group -> set of carrier UUIDs
        group_miss_pos  = defaultdict(set)   # source_group -> set of missense positions
        for miss_pos in gene_miss['pos'].unique():
            nearby = get_nearby(acc, int(miss_pos))
            if contact_pos not in nearby: continue
            matched = gene_miss[gene_miss['pos'] == miss_pos]
            for grp in matched['group'].dropna().unique():
                group_carriers[grp] |= set(matched.loc[matched['group'] == grp, 'sample_id'])
                group_miss_pos[grp].add(int(miss_pos))

        if not group_carriers: continue

        # Non-carriers = patients with NO nearby missense of any group
        all_carrier_uuids = set().union(*group_carriers.values())
        noncarrier_chs = [ch for ch in channels_with_genomics
                          if ch_uuid.get(ch) not in all_carrier_uuids and ch in ri_col_map]
        if len(noncarrier_chs) < 2: continue

        swap_row = pep_ri.loc[pep_seq]
        wt_row   = pep_ri.loc[wt_pep]

        # Emit one set of records per source group independently
        for source_group, carrier_uuids in group_carriers.items():
            carrier_chs = [ch for ch in channels_with_genomics
                           if ch_uuid.get(ch) in carrier_uuids and ch in ri_col_map]
            if not carrier_chs: continue

            for ch in carrier_chs + noncarrier_chs:
                swap_ri = swap_row.get(ch, 0)
                wt_ri   = wt_row.get(ch, 0)
                if wt_ri <= 0: continue
                raas = swap_ri / wt_ri
                log2_raas = float(np.log2(raas)) if raas > 0 else np.nan
                records.append({
                    'plex_id':      plex_id,
                    'gene':         gene,
                    'contact_pos':  contact_pos,
                    'swap':         f'{wt_aa}{contact_pos}{alt_aa}',
                    'wt_peptide':   wt_pep,
                    'channel':      ch,
                    'source_group': source_group,
                    'is_carrier':   ch in carrier_chs,
                    'swap_ri':      float(swap_ri),
                    'wt_ri':        float(wt_ri),
                    'log2_raas':    log2_raas,
                    'miss_positions': ','.join(str(p) for p in sorted(group_miss_pos[source_group])),
                })
    n_done += 1

df = pd.DataFrame(records)
print(f'Done: {n_done} plexes | {len(df):,} channel records | {n_no_wt} swap/no-WT skips')
if len(df):
    print(df['source_group'].value_counts())
    print(f"Carrier rows: {df['is_carrier'].sum():,} | Non-carrier: {(~df['is_carrier']).sum():,}")


In [ ]:
# ── FLAG SUSPICIOUS SWAPS ─────────────────────────────────────────────────────
# Swaps whose mass difference is within 0.05 Da of a common PTM mass may be
# modification artifacts rather than true amino acid substitutions.
# Use CLEAN_ONLY = True to restrict downstream analysis to unambiguous swaps.

_AA_MASS = {
    'A': 71.03711, 'C': 103.00919, 'D': 115.02694, 'E': 129.04259,
    'F': 147.06841, 'G':  57.02146, 'H': 137.05891, 'I': 113.08406,
    'K': 128.09496, 'L': 113.08406, 'M': 131.04049, 'N': 114.04293,
    'P':  97.05276, 'Q': 128.05858, 'R': 156.10111, 'S':  87.03203,
    'T': 101.04768, 'V':  99.06841, 'W': 186.07931, 'Y': 163.06333,
}
_PTM_MASSES = {
    'methylation':              14.01565,
    'oxidation':                15.99491,
    'dehydrogenation':           2.01565,
    'deamidation':               0.98402,
    'acetylation':              42.01057,
    'phosphorylation':          79.96633,
    'water_addition':           18.01056,
    'methylation+deamidation':  14.99967,
    'double_methylation':       28.03130,
    'oxidation+deamidation':    16.97893,
    'double_oxidation':         31.98983,
}
_PTM_TOL = 0.05  # Da

def _swap_suspicion(swap_str):
    """Return PTM name if swap mass difference is within tolerance, else None."""
    m = __import__('re').match(r'^([A-Z])(\d+)([A-Z])$', swap_str)
    if not m: return None
    wt, alt = m.group(1), m.group(3)
    if wt not in _AA_MASS or alt not in _AA_MASS: return None
    delta = abs(_AA_MASS[alt] - _AA_MASS[wt])
    for ptm, mass in _PTM_MASSES.items():
        if abs(delta - mass) < _PTM_TOL:
            return ptm
    return None

df['suspicion'] = df['swap'].apply(_swap_suspicion)
df['suspicious'] = df['suspicion'].notna()

print('Swap suspicion breakdown:')
print(df.drop_duplicates(['gene','contact_pos','swap'])[['swap','suspicion']]
        .groupby('suspicion').size().sort_values(ascending=False).to_string())
print(f"\nClean swaps: {(~df['suspicious']).sum():,} records")
print(f"Suspicious:  {df['suspicious'].sum():,} records")
print()
print('Suspicious swap types detected:')
print(df[df['suspicious']].drop_duplicates(['gene','contact_pos','swap'])
        .groupby('suspicion')['swap'].count().sort_values(ascending=False).to_string())



In [ ]:
# ── AGGREGATE TO ONE DELTA PER EVENT ──────────────────────────────────────────
# For each (plex, gene, swap): Δ = mean_carrier_log2(swap/WT) - mean_noncarrier_log2(swap/WT)
# This controls for any systematic swap/WT ratio differences unrelated to carrier status.

valid = df[df['log2_raas'].notna()]

def _delta(g):
    c  = g.loc[ g['is_carrier'], 'log2_raas']
    nc = g.loc[~g['is_carrier'], 'log2_raas']
    if len(c) == 0 or len(nc) == 0:
        return pd.Series({'delta': float('nan'), 'n_carrier': len(c), 'n_noncarrier': len(nc)})
    return pd.Series({'delta': c.mean() - nc.mean(),
                      'n_carrier': len(c), 'n_noncarrier': len(nc)})

event_df = (valid
    .groupby(['plex_id','gene','contact_pos','swap','source_group'])
    .apply(_delta)
    .reset_index()
    .dropna(subset=['delta'])
)

print(f'Events with delta: {len(event_df):,}')
print(event_df['source_group'].value_counts().to_string())
print()

DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']

print('── Δlog2(swap/WT):  carrier − non-carrier ──────────────────────────────')
for grp in DESTAB_GROUPS + NEUTRAL_GROUPS:
    d = event_df[event_df['source_group'] == grp]['delta']
    if len(d) < 3: continue
    t, p = stats.ttest_1samp(d, popmean=0)
    print(f'{grp:<20}  n={len(d):,}  mean={d.mean():.4f}  median={d.median():.4f}  p={p:.2e}')

destab_d  = event_df[event_df['source_group'].isin(DESTAB_GROUPS)]['delta']
neutral_d = event_df[event_df['source_group'].isin(NEUTRAL_GROUPS)]['delta']
if len(destab_d) >= 3 and len(neutral_d) >= 3:
    u, p_mw = stats.mannwhitneyu(destab_d, neutral_d, alternative='greater')
    print(f'\nDestab vs Neutral (MW, greater): p={p_mw:.2e}  '
          f'(n={len(destab_d):,} vs {len(neutral_d):,})')


In [ ]:
# ── PLOTS: violin + median bar, all swaps vs clean only ───────────────────────
import math

DESTAB_GROUPS  = ['Both AM+SPURS', 'SPURS only', 'AM only']
NEUTRAL_GROUPS = ['Neutral']
GROUP_SPECS = [
    ('Both AM+SPURS', '#d62728'),
    ('SPURS only',    '#ff7f0e'),
    ('AM only',       '#9467bd'),
    ('Neutral',       '#2ca02c'),
]

def make_event_df(use_clean):
    _df = df[~df['suspicious']] if use_clean else df
    valid = _df[_df['log2_raas'].notna()]
    def _delta(g):
        c  = g.loc[ g['is_carrier'], 'log2_raas']
        nc = g.loc[~g['is_carrier'], 'log2_raas']
        if len(c) == 0 or len(nc) == 0:
            return pd.Series({'delta': float('nan'), 'n_carrier': len(c), 'n_noncarrier': len(nc)})
        return pd.Series({'delta': c.mean() - nc.mean(),
                          'n_carrier': len(c), 'n_noncarrier': len(nc)})
    return (valid
        .groupby(['plex_id','gene','contact_pos','swap','source_group'])
        .apply(_delta)
        .reset_index()
        .dropna(subset=['delta']))

edf_all   = make_event_df(use_clean=False)
edf_clean = make_event_df(use_clean=True)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for row_i, (edf, label) in enumerate([(edf_all, 'All swaps'), (edf_clean, 'Clean swaps only')]):
    ax_vio = axes[row_i, 0]
    ax_bar = axes[row_i, 1]

    # Violin
    violin_data, violin_colors, xtick_labels = [], [], []
    for grp, color in GROUP_SPECS:
        d = edf[edf['source_group'] == grp]['delta']
        if len(d) < 3: continue
        violin_data.append(d.values)
        violin_colors.append(color)
        xtick_labels.append(f'{grp}\nn={len(d):,}')
    if violin_data:
        parts = ax_vio.violinplot(violin_data, positions=range(len(violin_data)),
                                  showmedians=True, showextrema=False)
        for pc, col in zip(parts['bodies'], violin_colors):
            pc.set_facecolor(col); pc.set_alpha(0.7)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
        for i, d in enumerate(violin_data):
            ax_vio.scatter([i]*len(d), d, color='k', alpha=0.25, s=6, zorder=3)
    ax_vio.axhline(0, color='k', lw=0.8, ls='--')
    ax_vio.set_xticks(range(len(xtick_labels)))
    ax_vio.set_xticklabels(xtick_labels, fontsize=8)
    ax_vio.set_ylabel('Δ log2(swap/WT): carrier − non-carrier')
    ax_vio.set_title(f'{label} — distribution (median line)')

    # Median bar with IQR error bars (consistent with violin median line)
    bar_i = 0
    for grp, color in GROUP_SPECS:
        d = edf[edf['source_group'] == grp]['delta'].dropna()
        if len(d) == 0: continue
        med = d.median()
        q1, q3 = d.quantile(0.25), d.quantile(0.75)
        t, p = stats.ttest_1samp(d, popmean=0) if len(d) >= 3 else (float('nan'), float('nan'))
        ax_bar.bar(bar_i, med, color=color, alpha=0.85)
        ax_bar.errorbar(bar_i, med, yerr=[[med - q1], [q3 - med]],
                        fmt='none', color='black', capsize=5, linewidth=1.5)
        pstr = f'p={p:.3f}' if not math.isnan(p) else ''
        ax_bar.text(bar_i, q3 + 0.02,
                    f'n={len(d):,}\n{pstr}', ha='center', va='bottom', fontsize=7.5)
        bar_i += 1
    ax_bar.axhline(0, color='k', lw=0.8, ls='--')
    ax_bar.set_xticks(range(bar_i))
    ax_bar.set_xticklabels([g[0] for g in GROUP_SPECS[:bar_i]], fontsize=8, rotation=15, ha='right')
    ax_bar.set_ylabel('median Δ log2(swap/WT)  [IQR error bars]')
    ax_bar.set_title(f'{label} — median + IQR (t-test vs 0)')

plt.suptitle('Contact-predicted SAAP: carrier vs non-carrier RAAS delta\n'
             'Δ = mean_carrier[log2(swap/WT)] − mean_noncarrier[log2(swap/WT)]',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_raas_delta.pdf'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
import re as _re2
issues = []
n_checked = 0

for (gene, contact_pos, swap), grp in df.groupby(['gene','contact_pos','swap']):
    row0 = grp.iloc[0]
    wt_pep  = row0['wt_peptide']
    pep_seq = None  # swap peptide seq not stored — checked via swap string

    # Parse wt_aa and alt_aa from swap string e.g. "K245A"
    m = _re2.match(r'^([A-Z])(\d+)([A-Z])$', swap)
    if not m:
        issues.append(f'Unparseable swap string: {swap} in {gene}')
        continue
    wt_aa, pos, alt_aa = m.group(1), int(m.group(2)), m.group(3)

    acc = gene_to_acc.get(gene)
    seq = seqs.get(acc, '') if acc else ''

    # 1. contact_pos within protein length
    if seq and (contact_pos < 1 or contact_pos > len(seq)):
        issues.append(f'{gene} {swap}: contact_pos {contact_pos} out of range (len={len(seq)})')
        continue

    # 2. wt_aa matches reference sequence at contact_pos
    if seq and seq[contact_pos - 1] != wt_aa:
        issues.append(f'{gene} {swap}: wt_aa={wt_aa} but ref has {seq[contact_pos-1]} at pos {contact_pos}')

    # 3. wt_peptide should contain wt_aa somewhere (and NOT alt_aa at that position)
    if seq and wt_pep:
        # Find where wt_pep maps in the protein to get the contact position within peptide
        pep_start = seq.find(wt_pep)
        if pep_start == -1:
            issues.append(f'{gene} {swap}: wt_peptide not found in reference sequence')
        else:
            pep_idx = contact_pos - 1 - pep_start
            if 0 <= pep_idx < len(wt_pep):
                if wt_pep[pep_idx] != wt_aa:
                    issues.append(f'{gene} {swap}: wt_peptide has {wt_pep[pep_idx]} at contact pos, expected {wt_aa}')
                if wt_pep[pep_idx] == alt_aa:
                    issues.append(f'{gene} {swap}: wt_peptide already contains alt_aa {alt_aa} at contact pos!')

    # 4. Self-swap
    if wt_aa == alt_aa:
        issues.append(f'{gene} {swap}: wt_aa == alt_aa ({wt_aa})')

    # 5. Excluded swap slipped through
    excluded = {('N','D'),('D','N'),('Q','E'),('E','Q'),('I','L'),('L','I')}
    if (wt_aa, alt_aa) in excluded:
        issues.append(f'{gene} {swap}: excluded swap ({wt_aa}->{alt_aa}) found in results')
    if wt_aa in 'KR' or alt_aa in 'KR':
        issues.append(f'{gene} {swap}: K/R swap found in results')

    n_checked += 1

print(f'Checked {n_checked} unique (gene, contact_pos, swap) combinations')
if issues:
    print(f'\n{len(issues)} issues found:')
    for iss in issues[:50]:
        print(f'  {iss}')
    if len(issues) > 50:
        print(f'  ... and {len(issues)-50} more')
    # 7. Swap peptide sequence matches a canonical human tryptic peptide
    #    (check via wt_peptide with alt_aa substituted back in)
    if seq and wt_pep:
        pep_start = seq.find(wt_pep)
        if pep_start >= 0:
            pep_idx = contact_pos - 1 - pep_start
            if 0 <= pep_idx < len(wt_pep):
                swap_pep = wt_pep[:pep_idx] + alt_aa + wt_pep[pep_idx+1:]
                if swap_pep in canonical_peptides:
                    issues.append(f'{gene} {swap}: swap peptide {swap_pep} exists as canonical human peptide!')

else:
    print('No issues found.')


In [ ]:
# ── SWAP HEATMAP: from AA (x) → to AA (y), color = unique swap peptides ───────
import re as _re3

AA_ORDER = list('ACDEFGHIKLMNPQRSTVWY')

# Parse wt_aa and alt_aa from swap column, count unique (gene, contact_pos, swap)
swap_counts = (df.drop_duplicates(['gene','contact_pos','swap'])
                 ['swap']
                 .str.extract(r'^([A-Z])\d+([A-Z])$')
                 .rename(columns={0:'wt', 1:'alt'})
                 .dropna()
                 .groupby(['wt','alt'])
                 .size()
                 .reset_index(name='n'))

# Build 20x20 matrix
mat = pd.DataFrame(0, index=AA_ORDER, columns=AA_ORDER)
for _, row in swap_counts.iterrows():
    if row['wt'] in mat.index and row['alt'] in mat.columns:
        mat.loc[row['wt'], row['alt']] = row['n']

import numpy as np
mat_np = mat.values.astype(float)
mat_np[mat_np == 0] = np.nan  # grey for undetected

fig, ax = plt.subplots(figsize=(10, 9))
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad('lightgrey')

im = ax.imshow(mat_np, cmap=cmap, aspect='equal')
plt.colorbar(im, ax=ax, label='# unique swap peptides detected')

ax.set_xticks(range(len(AA_ORDER)))
ax.set_yticks(range(len(AA_ORDER)))
ax.set_xticklabels(AA_ORDER, fontsize=9)
ax.set_yticklabels(AA_ORDER, fontsize=9)
ax.set_xlabel('From (WT amino acid)', fontsize=11)
ax.set_ylabel('To (swap amino acid)', fontsize=11)
ax.set_title('Contact-predicted SAAP detections\nby amino acid swap type', fontsize=12)

# Annotate cells with count where > 0
for i, wt in enumerate(AA_ORDER):
    for j, alt in enumerate(AA_ORDER):
        v = mat.loc[wt, alt]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=6, color='black' if v < mat_np[~np.isnan(mat_np)].max()*0.6 else 'white')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_swap_heatmap.pdf'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Total unique swap peptides: {df.drop_duplicates(['gene','contact_pos','swap']).shape[0]:,}")
print("\nTop swap types:")
print(swap_counts.sort_values('n', ascending=False).head(10).to_string(index=False))


In [ ]:
# ── PSM CONFIDENCE SCORES + LOCALIZATION PROXY ────────────────────────────────
# Re-reads psm.tsv for contact PSMs to get:
#   PEP (1 - PeptideProphet prob), hyperscore, matched/total ion fraction
# Localization proxy: is the swap position flanked by >=2 theoretical ions
#   on each side? (i.e. >= 2 b-ions N-terminal and >= 2 y-ions C-terminal)
# This is a necessary (not sufficient) condition for unambiguous localization.

score_records = []
for plex_id in plex_ids:
    psm_files = sorted(glob.glob(os.path.join(RESULTS_BASE, plex_id, '*_1/psm.tsv')))
    if not psm_files: continue
    try:
        psm = pd.read_csv(psm_files[0], sep='\t', low_memory=False)
    except Exception: continue

    if not score_records:  # print column names once from first plex
        score_cols = [c for c in psm.columns if any(k in c.lower()
                      for k in ['prob','hyper','match','ion','expect','score','pep'])]
        print(f"PSM score columns available: {score_cols}")
    contact_mask = psm['Entry Name'].astype(str).str.endswith('-contact')
    contact_psm  = psm[contact_mask].copy()
    if len(contact_psm) == 0: continue

    for _, row in contact_psm.iterrows():
        entry_name = str(row.get('Entry Name', ''))
        protein_id = str(row.get('Protein ID', ''))
        pep_seq    = str(row.get('Peptide', ''))

        m_gene = _re.match(r'^([A-Z0-9]+)-contact$', entry_name)
        m_swap = _re.search(r'-([A-Z])(\d+)([A-Z])-[A-Z0-9]{4}$', protein_id)
        if not m_gene or not m_swap: continue

        gene = m_gene.group(1)
        wt_aa, contact_pos, alt_aa = m_swap.group(1), int(m_swap.group(2)), m_swap.group(3)

        # PSM confidence scores — try multiple FragPipe column name variants
        def _get(*keys):
            for k in keys:
                v = row.get(k, np.nan)
                if pd.notna(v): return pd.to_numeric(v, errors='coerce')
            return np.nan
        prob       = _get('PeptideProphet Probability', 'Probability')
        pep        = float(1 - prob) if pd.notna(prob) else np.nan
        hyperscore = _get('Hyperscore', 'MSFragger:Hyperscore', 'hyperscore')
        n_matched  = _get('Number of Matched Ions', 'Matched Ions', 'Matched Fragment Ions')
        n_total    = _get('Total Number of Ions', 'Total Ions', 'Total Fragment Ions')
        matched_frac = float(n_matched / n_total) if (pd.notna(n_matched) and pd.notna(n_total) and n_total > 0) else np.nan
        expectation  = _get('Expectation', 'MSFragger:Expectation', 'e-value')

        # Localization proxy: find swap position within peptide
        acc = gene_to_acc.get(gene)
        seq = seqs.get(acc, '') if acc else ''
        wt_peps = get_wt_peptides(acc, contact_pos) if acc else []
        wt_pep  = next((p for p in wt_peps if seq.find(p) >= 0), None)

        swap_pos_in_pep = np.nan
        localizable = False
        if wt_pep and seq:
            pep_start = seq.find(wt_pep)
            if pep_start >= 0:
                swap_pos_in_pep = (contact_pos - 1) - pep_start  # 0-indexed within peptide
                n_left  = int(swap_pos_in_pep)                   # b-ions not containing swap
                n_right = len(wt_pep) - int(swap_pos_in_pep) - 1 # y-ions not containing swap
                localizable = min(n_left, n_right) >= 2

        score_records.append({
            'plex_id':         plex_id,
            'gene':            gene,
            'contact_pos':     contact_pos,
            'swap':            f'{wt_aa}{contact_pos}{alt_aa}',
            'wt_peptide':      wt_pep or '',
            'swap_peptide':    pep_seq,
            'pep':             pep,
            'hyperscore':      float(hyperscore) if pd.notna(hyperscore) else np.nan,
            'matched_frac':    matched_frac,
            'expectation':     float(expectation) if pd.notna(expectation) else np.nan,
            'swap_pos_in_pep': swap_pos_in_pep,
            'localizable':     localizable,
        })

scores_df = pd.DataFrame(score_records)
print(f"Contact PSMs: {len(scores_df):,} across {scores_df['plex_id'].nunique()} plexes")
print(f"Localizable (>=2 ions each side of swap): {scores_df['localizable'].sum():,} / {len(scores_df):,} "
      f"({100*scores_df['localizable'].mean():.1f}%)")
print(f"Median matched ion fraction: {scores_df['matched_frac'].median():.3f}")
print(f"Median PEP: {scores_df['pep'].median():.4f}")
print(f"\nBest-confidence PSMs (PEP<0.01, localizable, matched_frac>0.3):")
hi = scores_df[scores_df['localizable'] & (scores_df['pep'] < 0.01) & (scores_df['matched_frac'] > 0.3)]
print(hi[['gene','swap','pep','hyperscore','matched_frac']].drop_duplicates(['gene','swap'])
        .sort_values('pep').head(20).to_string(index=False))


In [ ]:
# ── FINAL SUMMARY TABLE ────────────────────────────────────────────────────────
# One row per (gene, swap, source_group) aggregated across all plexes.
# Columns: peptides, carrier/noncarrier channels, missense scores,
#          plex counts, PSM confidence, localization proxy.

# 1. RAAS delta per event (from event_df computed in stats cell)
event_stats = (event_df
    .groupby(['gene','contact_pos','swap','source_group'])
    .agg(
        n_events        = ('delta', 'count'),
        median_delta    = ('delta', 'median'),
        mean_delta      = ('delta', 'mean'),
        n_carrier_ch    = ('n_carrier',    'sum'),
        n_noncarrier_ch = ('n_noncarrier', 'sum'),
    )
    .reset_index())

# 2. PSM confidence scores per (gene, swap) — best across all PSMs
score_stats = (scores_df
    .groupby(['gene','contact_pos','swap'])
    .agg(
        swap_peptide    = ('swap_peptide', 'first'),
        wt_peptide      = ('wt_peptide',   'first'),
        best_pep        = ('pep',          'min'),
        best_hyperscore = ('hyperscore',   'max'),
        median_matched_frac = ('matched_frac', 'median'),
        n_psms          = ('pep',          'count'),
        n_plexes_detected = ('plex_id',    'nunique'),
        pct_localizable = ('localizable',  'mean'),
    )
    .reset_index())

# 3. Missense scores — for each (gene, contact_pos, source_group)
#    look up the nearby missense in the full missense table
miss_stats_records = []
for (gene, cpos, grp), grp_df in df.groupby(['gene','contact_pos','source_group']):
    acc = gene_to_acc.get(gene)
    if not acc: continue
    gene_miss = all_miss[
        (all_miss['SYMBOL'] == gene) &
        (all_miss['group']  == grp)
    ].dropna(subset=['pos'])
    nearby_miss = gene_miss[[cpos in get_nearby(acc, int(p))
                              for p in gene_miss['pos']]]
    if len(nearby_miss) == 0: continue
    miss_stats_records.append({
        'gene':            gene,
        'contact_pos':     cpos,
        'source_group':    grp,
        'missense_am':     nearby_miss['am_pathogenicity'].median(),
        'missense_spurs':  nearby_miss['spurs_ddg'].median(),
        'missense_vaf':    nearby_miss['VAF'].median(),
        'missense_gnomad': nearby_miss['gnomADe_AF'].median(),
        'n_patients_with_missense': nearby_miss['sample_id'].nunique(),
    })
miss_stats = pd.DataFrame(miss_stats_records)

# 4. Number of plexes where the missense exists (regardless of detection)
plex_miss_counts = (all_miss[all_miss['SYMBOL'].isin(df['gene'].unique())]
    .groupby('SYMBOL')['sample_id']
    .nunique()
    .rename('n_patients_total')
    .reset_index()
    .rename(columns={'SYMBOL':'gene'}))

# 5. Suspicious flag per swap
sus_map = (df[['swap','suspicious','suspicion']]
           .drop_duplicates('swap')
           .set_index('swap'))

# Merge everything
summary = (event_stats
    .merge(score_stats,  on=['gene','contact_pos','swap'], how='left')
    .merge(miss_stats,   on=['gene','contact_pos','source_group'], how='left')
    .merge(plex_miss_counts, on='gene', how='left'))
summary['suspicious'] = summary['swap'].map(sus_map['suspicious']).fillna(False)
summary['suspicion']  = summary['swap'].map(sus_map['suspicion'])

# Sort by best_pep then median_delta descending
summary = summary.sort_values(['best_pep','median_delta'], ascending=[True, False])

cols = ['gene','swap','swap_peptide','wt_peptide','source_group',
        'n_events','n_carrier_ch','n_noncarrier_ch',
        'median_delta','mean_delta',
        'missense_am','missense_spurs','missense_vaf','missense_gnomad',
        'n_patients_with_missense','n_patients_total',
        'n_plexes_detected','n_psms',
        'best_pep','best_hyperscore','median_matched_frac','pct_localizable',
        'suspicious','suspicion']
summary = summary[[c for c in cols if c in summary.columns]]

# Filter Ig genes — somatic hypermutation makes swaps uninterpretable
IG_PREFIXES = ('IGH','IGL','IGK','IGHA','IGHG','IGHM','IGLC','IGKC')
ig_mask = summary['gene'].str.startswith(IG_PREFIXES)
summary_clean = summary[~ig_mask].copy()
print(f"Summary table: {len(summary):,} rows total, {ig_mask.sum()} Ig rows removed")
print(f"Clean summary: {len(summary_clean):,} rows")

print("\nTop 20 clean hits (localizable, non-suspicious, non-Ig):")
top = summary_clean[~summary_clean['suspicious'] & (summary_clean['pct_localizable'] > 0.5)]
print(top.head(20)[['gene','swap','source_group','n_events','median_delta',
                      'best_pep','pct_localizable','missense_am','missense_spurs']].to_string(index=False))

summary.to_csv(os.path.join(OUT_DIR, 'contact_saap_summary.tsv'), sep='\t', index=False)
summary_clean.to_csv(os.path.join(OUT_DIR, 'contact_saap_summary_clean.tsv'), sep='\t', index=False)
print(f"\nSaved to {OUT_DIR}/contact_saap_summary*.tsv")


In [ ]:
# ── MULTI-SWAP, RECURRENCE, PROTEIN-LEVEL AAS ANALYSIS ───────────────────────

# ── 1. Multiple swap types at same contact site in same plex ──────────────────
# If contact_pos has >1 different alt_aa detected in the same plex, this is
# strong evidence the site is genuinely prone to misincorporation.
multi = (scores_df
    .groupby(['plex_id','gene','contact_pos'])['swap']
    .nunique()
    .reset_index(name='n_swap_types'))
multi_sites = multi[multi['n_swap_types'] > 1].sort_values('n_swap_types', ascending=False)
print(f"Contact sites with >1 swap type in same plex: {len(multi_sites):,}")
print(multi_sites.head(20).to_string(index=False))

# Across all plexes: sites with multiple swap types
multi_global = (scores_df
    .groupby(['gene','contact_pos'])
    .agg(n_swap_types=('swap','nunique'),
         n_plexes=('plex_id','nunique'),
         swaps_detected=('swap', lambda x: ', '.join(sorted(x.unique()))))
    .reset_index()
    .sort_values('n_swap_types', ascending=False))
print(f"\nGlobally: {(multi_global['n_swap_types']>1).sum()} contact sites with >1 swap type")
print(multi_global[multi_global['n_swap_types']>1].head(20).to_string(index=False))

# ── 2. Carrier patient recurrence: unique patients per (gene, swap) ───────────
# Count unique carrier patients across all plexes for each swap
carrier_patients = (df[df['is_carrier']]
    .merge(pd.DataFrame({'plex_id': plex_ids})
           .assign(tmp=1)
           .merge(
               tmt[['run_metadata_id','case_submitter_id']].rename(columns={'run_metadata_id':'plex_id'}),
               on='plex_id', how='left'),
           on='plex_id', how='left')
    if 'case_submitter_id' in tmt.columns else df[df['is_carrier']])

# Simpler: count unique plex×channel combinations as proxy for patients
patient_recurrence = (df[df['is_carrier']]
    .groupby(['gene','swap','source_group'])
    .agg(n_carrier_observations=('channel','count'),
         n_plexes_as_carrier=('plex_id','nunique'))
    .reset_index()
    .sort_values('n_plexes_as_carrier', ascending=False))
print(f"\n── Patient recurrence of carrier-swap pairs ─────────────────────────")
print(patient_recurrence.head(20).to_string(index=False))

# ── 3. Load Tsour et al. and get protein-level AAS burden ────────────────────
TSOUR_PEP2PROT = f'{REPO_DIR}/metadata/Tsour_et_al/pep_to_protein.csv'
TSOUR_PEP2PAT  = f'{REPO_DIR}/metadata/Tsour_et_al/peptide_to_patient.csv'
try:
    pep2prot = pd.read_csv(TSOUR_PEP2PROT)
    pep2pat  = pd.read_csv(TSOUR_PEP2PAT)

    # Map SAAP gene to proteins in our detected swap set
    detected_genes = set(scores_df['gene'].unique())
    # pep2prot column for gene symbol
    gene_col = next((c for c in pep2prot.columns
                     if c.lower() in ['gene','symbol','gene_symbol','name']), pep2prot.columns[0])
    tsour_gene_aas = pep2prot[pep2prot[gene_col].isin(detected_genes)]
    print(f"\n── Tsour et al. AAS on proteins with detected contact swaps ─────────")
    print(f"Genes in our contact swap set that have Tsour AAS: "
          f"{tsour_gene_aas[gene_col].nunique()} / {len(detected_genes)}")
    print(tsour_gene_aas[gene_col].value_counts().head(15).to_string())
except Exception as e:
    print(f"\nCould not load Tsour data: {e}")

# ── 4. Protein list: all genes with contact-predicted SAAP + missense ─────────
protein_list = (summary_clean
    .groupby('gene')
    .agg(n_swaps_detected=('swap','nunique'),
         n_contact_sites=('contact_pos','nunique'),
         best_pep=('best_pep','min'),
         best_delta=('median_delta','max'),
         source_groups=('source_group', lambda x: ', '.join(sorted(x.unique()))))
    .reset_index()
    .sort_values('n_swaps_detected', ascending=False))
print(f"\n── Proteins with contact-predicted SAAP detections ─────────────────")
print(f"Total proteins: {len(protein_list):,}")
print(protein_list.head(30).to_string(index=False))
protein_list.to_csv(os.path.join(OUT_DIR, 'contact_saap_protein_list.tsv'), sep='\t', index=False)


In [ ]:
# ── MISSENSE → MULTI-SITE AND MULTI-SWAP ANALYSIS ────────────────────────────
# Each missense at position i can drive swaps at multiple contact positions j,
# and each contact position j can have multiple detected swap amino acids.

# Expand miss_positions to one row per missense position
df_miss = df.copy()
df_miss = df_miss[df_miss['miss_positions'].notna() & (df_miss['miss_positions'] != '')]
df_miss = df_miss.assign(miss_pos=df_miss['miss_positions'].str.split(','))
df_miss = df_miss.explode('miss_pos')
df_miss['miss_pos'] = pd.to_numeric(df_miss['miss_pos'], errors='coerce')

# ── 1. Per missense: how many unique contact sites had detected swaps ──────────
miss_to_contacts = (df_miss
    .drop_duplicates(['plex_id','gene','miss_pos','contact_pos'])
    .groupby(['gene','miss_pos','source_group'])
    .agg(n_contact_sites=('contact_pos','nunique'),
         contact_positions=('contact_pos', lambda x: ','.join(str(v) for v in sorted(x.unique()))),
         n_plexes=('plex_id','nunique'))
    .reset_index()
    .sort_values('n_contact_sites', ascending=False))

print("── Missense positions driving multiple contact sites ─────────────────")
multi_contact = miss_to_contacts[miss_to_contacts['n_contact_sites'] > 1]
print(f"{len(multi_contact):,} missense positions drive >1 detected contact site")
print(multi_contact.head(20).to_string(index=False))

# ── 2. Per contact site: how many unique swap amino acids detected ─────────────
contact_to_swaps = (df_miss
    .drop_duplicates(['gene','contact_pos','swap'])
    .groupby(['gene','contact_pos','source_group'])
    .agg(n_swap_types=('swap','nunique'),
         swap_aas=('swap', lambda x: ','.join(sorted(x.unique()))),
         n_plexes=('plex_id','nunique'),
         n_driving_missense=('miss_pos','nunique'))
    .reset_index()
    .sort_values('n_swap_types', ascending=False))

print("\n── Contact sites with multiple detected swap amino acids ─────────────")
multi_swap = contact_to_swaps[contact_to_swaps['n_swap_types'] > 1]
print(f"{len(multi_swap):,} contact sites with >1 swap type detected")
print(multi_swap.head(20).to_string(index=False))

# ── 3. Protein-level summary ───────────────────────────────────────────────────
protein_summary = (df_miss
    .drop_duplicates(['gene','miss_pos','contact_pos','swap'])
    .groupby('gene')
    .agg(n_missense_positions=('miss_pos','nunique'),
         n_contact_sites=('contact_pos','nunique'),
         n_swap_types=('swap','nunique'),
         n_plexes=('plex_id','nunique'),
         source_groups=('source_group', lambda x: ','.join(sorted(x.unique()))))
    .reset_index()
    .sort_values('n_swap_types', ascending=False))

# Filter Ig
ig_mask = protein_summary['gene'].str.startswith(('IGH','IGL','IGK','IGHA','IGHG','IGHM','IGLC','IGKC'))
protein_summary_clean = protein_summary[~ig_mask]

print("\n── Protein list: missense-driven contact SAAP detections ─────────────")
print(f"{len(protein_summary_clean):,} proteins (excluding Ig)")
print(protein_summary_clean.head(30).to_string(index=False))

protein_summary_clean.to_csv(os.path.join(OUT_DIR, 'contact_saap_protein_summary.tsv'), sep='\t', index=False)
miss_to_contacts.to_csv(os.path.join(OUT_DIR, 'contact_saap_missense_to_contacts.tsv'), sep='\t', index=False)
print(f"\nSaved to {OUT_DIR}/")


In [ ]:
# ── COMPLEMENTARITY ANALYSIS ──────────────────────────────────────────────────
# Classify each (missense, swap) pair by whether the swap at contact position j
# is chemically complementary to the missense at position i.
#
# Rules applied:
#   Charge reversal: K/R->D/E at i => complementary swap is D/E->K/R at j
#                    D/E->K/R at i => complementary swap is K/R->D/E at j
#   Charge burial:   hydrophobic->charged at i, contact j had same-sign charge
#                    => complementary swap removes that charge at j
#   Size packing:    large->small at i (cavity) => complementary swap is
#                    small->large at j (fills cavity), and vice versa

import math as _math

_POS  = {'K','R','H'}
_NEG  = {'D','E'}
_CHRG = _POS | _NEG
_PHOB = {'A','V','I','L','M','F','W','P'}
_AA_VOL = {
    'G':60,'A':89,'S':96,'C':108,'P':113,'T':116,'V':140,'D':111,'N':114,
    'L':164,'I':166,'E':138,'Q':143,'M':163,'H':153,'F':190,'R':173,
    'Y':193,'W':228,'K':169
}
_FAVORABLE = {'salt_bridge','hydrophobic'}

def _interaction_type(aa1, aa2):
    """Classify the pairwise interaction type between two residues."""
    if not aa1 or not aa2: return 'unknown'
    if (aa1 in _POS and aa2 in _NEG) or (aa1 in _NEG and aa2 in _POS):
        return 'salt_bridge'
    if (aa1 in _POS and aa2 in _POS) or (aa1 in _NEG and aa2 in _NEG):
        return 'charge_repulsion'
    if aa1 in _PHOB and aa2 in _PHOB:
        return 'hydrophobic'
    if aa1 in _CHRG or aa2 in _CHRG:
        return 'polar_charged'
    return 'polar_neutral'

def _classify(wt_i, alt_i, wt_j, alt_j):
    """Three-step analysis anchored to the original WT-WT interaction.
    
    Returns dict with:
      wt_interaction      : interaction type in the original protein
      missense_effect     : broken / maintained / changed
      post_swap_interaction: interaction type after both missense and swap
      complementarity     : canonical_restoration / compensatory /
                            neutral / disruptive
    """
    if not all(a in _AA_VOL for a in [wt_i, alt_i, wt_j, alt_j]):
        return {'wt_interaction':'unknown','missense_effect':'unknown',
                'post_swap_interaction':'unknown','complementarity':'unknown'}

    wt_iact   = _interaction_type(wt_i,  wt_j)   # WT-WT
    miss_iact = _interaction_type(alt_i, wt_j)   # after missense, before swap
    swap_iact = _interaction_type(alt_i, alt_j)  # after both

    # Was WT interaction favorable?
    wt_fav  = wt_iact  in _FAVORABLE
    mis_fav = miss_iact in _FAVORABLE
    swp_fav = swap_iact in _FAVORABLE

    # Did missense break/maintain the interaction?
    if miss_iact == wt_iact:
        miss_effect = 'maintained'
    elif wt_fav and not mis_fav:
        miss_effect = 'broken'
    elif not wt_fav and mis_fav:
        miss_effect = 'improved'
    else:
        miss_effect = 'changed'

    # Classify the swap relative to the WT interaction context
    if miss_effect == 'broken' and swap_iact == wt_iact:
        comp = 'canonical_restoration'   # restores exact original interaction type
    elif miss_effect == 'broken' and swp_fav and not mis_fav:
        comp = 'compensatory'            # restores a favorable contact, not identical
    elif miss_effect == 'maintained' and swp_fav:
        comp = 'neutral'
    elif swap_iact == 'charge_repulsion' or (wt_fav and not swp_fav):
        comp = 'disruptive'
    else:
        comp = 'neutral'

    # Size-based override for cases not covered by charge/hydrophobic rules
    dv_i = _AA_VOL[alt_i] - _AA_VOL[wt_i]
    dv_j = _AA_VOL[alt_j] - _AA_VOL[wt_j]
    if comp == 'neutral':
        if   dv_i >  30 and dv_j < -30: comp = 'compensatory'
        elif dv_i < -30 and dv_j >  30: comp = 'compensatory'
        elif (dv_i >  30 and dv_j >  30) or (dv_i < -30 and dv_j < -30):
            comp = 'disruptive'

    return {'wt_interaction': wt_iact, 'missense_effect': miss_effect,
            'post_swap_interaction': swap_iact, 'complementarity': comp}

# Build missense AA change lookup: (gene, miss_pos) -> (wt_aa, alt_aa)
miss_aa = {}
for _, row in all_miss.drop_duplicates(['SYMBOL','pos','Amino_acids']).iterrows():
    aa = str(row.get('Amino_acids',''))
    if '/' in aa:
        wt_aa_i, alt_aa_i = aa.split('/')[0].strip(), aa.split('/')[1].strip()
        if len(wt_aa_i)==1 and len(alt_aa_i)==1:
            miss_aa[(str(row['SYMBOL']), int(row['pos']))] = (wt_aa_i, alt_aa_i)

# Classify each (gene, miss_pos, swap) combination
comp_records = []
for _, row in df_miss.drop_duplicates(['gene','miss_pos','contact_pos','swap']).iterrows():
    gene     = row['gene']
    miss_pos = row['miss_pos']
    swap     = row['swap']
    m = _re.match(r'^([A-Z])(\d+)([A-Z])$', swap)
    if not m or pd.isna(miss_pos): continue
    wt_j, alt_j = m.group(1), m.group(3)
    wt_i, alt_i  = miss_aa.get((gene, int(miss_pos)), (None, None))
    if wt_i is None: continue
    cl = _classify(wt_i, alt_i, wt_j, alt_j)
    comp_records.append({
        'gene': gene, 'miss_pos': miss_pos, 'contact_pos': row['contact_pos'],
        'swap': swap, 'miss_change': f'{wt_i}{int(miss_pos)}{alt_i}',
        'wt_i': wt_i, 'alt_i': alt_i, 'wt_j': wt_j, 'alt_j': alt_j,
        'wt_interaction':       cl['wt_interaction'],
        'missense_effect':      cl['missense_effect'],
        'post_swap_interaction':cl['post_swap_interaction'],
        'complementarity':      cl['complementarity'],
        'source_group': row['source_group'],
    })
comp_df = pd.DataFrame(comp_records)
print("Complementarity classification:")
print(comp_df['complementarity'].value_counts().to_string())
print("\nWT interaction types:")
print(comp_df['wt_interaction'].value_counts().to_string())
print("\nMissense effect on WT interaction:")
print(comp_df['missense_effect'].value_counts().to_string())
print("\nCanonical restorations (broken WT interaction restored by swap):")
canon = comp_df[comp_df['complementarity']=='canonical_restoration']
print(canon[['gene','miss_change','swap','wt_interaction']].drop_duplicates().to_string(index=False))

# Merge complementarity back onto df records
df_comp = df_miss.merge(
    comp_df[['gene','miss_pos','contact_pos','swap','miss_change','complementarity']],
    on=['gene','miss_pos','contact_pos','swap'], how='left')

# Aggregate delta per event with complementarity label
def _delta(g):
    c  = g.loc[ g['is_carrier'], 'log2_raas']
    nc = g.loc[~g['is_carrier'], 'log2_raas']
    if len(c) == 0 or len(nc) == 0:
        return pd.Series({'delta': float('nan')})
    return pd.Series({'delta': c.mean() - nc.mean()})

comp_events = (df_comp[df_comp['log2_raas'].notna()]
    .groupby(['gene','miss_pos','contact_pos','swap','source_group','complementarity'])
    .apply(_delta)
    .reset_index()
    .dropna(subset=['delta']))

print(f"\nEvents with complementarity label: {len(comp_events):,}")
print("\n── Median delta by complementarity (destab groups) ──────────────────")
destab = comp_events[comp_events['source_group'].isin(['Both AM+SPURS','SPURS only','AM only'])]
for comp_class in ['complementary','neutral','disruptive','unknown']:
    d = destab[destab['complementarity'] == comp_class]['delta'].dropna()
    if len(d) < 3: continue
    t, p = stats.ttest_1samp(d, popmean=0)
    u_comp = destab[destab['complementarity']=='complementary']['delta'].dropna()
    if len(u_comp) >= 3 and comp_class != 'complementary':
        _, p_vs_comp = stats.mannwhitneyu(u_comp, d, alternative='greater')
        print(f"  {comp_class:<15} n={len(d):>4}  median={d.median():+.3f}  "
              f"t-test p={p:.3f}  vs complementary p={p_vs_comp:.3f}")
    else:
        print(f"  {comp_class:<15} n={len(d):>4}  median={d.median():+.3f}  "
              f"t-test p={p:.3f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
classes = ['complementary','neutral','disruptive']
colors  = ['#2ca02c','#7f7f7f','#d62728']
for ax, grp_label, mask in [
        (axes[0], 'Destab groups',
         destab),
        (axes[1], 'Neutral group',
         comp_events[comp_events['source_group']=='Neutral'])]:
    vdata, vlabels, vcolors = [], [], []
    for cls, col in zip(classes, colors):
        d = mask[mask['complementarity']==cls]['delta'].dropna()
        if len(d) < 3: continue
        vdata.append(d.values); vlabels.append(f'{cls}\nn={len(d)}'); vcolors.append(col)
    if not vdata: continue
    parts = ax.violinplot(vdata, positions=range(len(vdata)),
                          showmedians=True, showextrema=False)
    for pc, col in zip(parts['bodies'], vcolors):
        pc.set_facecolor(col); pc.set_alpha(0.7)
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
    for i, d in enumerate(vdata):
        ax.scatter([i]*len(d), d, color='k', alpha=0.2, s=6)
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(range(len(vlabels))); ax.set_xticklabels(vlabels, fontsize=9)
    ax.set_ylabel('Δ log2(swap/WT): carrier − non-carrier')
    ax.set_title(f'{grp_label}: RAAS delta by\nswap-missense complementarity')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_complementarity.pdf'), dpi=150, bbox_inches='tight')
plt.show()

# Top complementary hits
print("\nTop complementary swap-missense pairs (by delta):")
top_comp = (comp_events[
    (comp_events['complementarity']=='complementary') &
    (comp_events['source_group'].isin(['Both AM+SPURS','SPURS only','AM only']))]
    .sort_values('delta', ascending=False)
    [['gene','miss_change','swap','source_group','delta','complementarity']]
    .head(20))
print(top_comp.to_string(index=False))
comp_events.to_csv(os.path.join(OUT_DIR, 'contact_saap_complementarity.tsv'), sep='\t', index=False)


# ── FACETED RAAS DELTA: destab groups × complementarity ──────────────────────
GROUP_SPECS  = [('Both AM+SPURS','#d62728'),('SPURS only','#ff7f0e'),
                ('AM only','#9467bd'),('Neutral','#2ca02c')]
COMP_CLASSES = [('canonical_restoration','solid'),('compensatory','dashed'),
                ('neutral','dotted'),('disruptive','dashdot')]

fig, axes = plt.subplots(1, len(GROUP_SPECS), figsize=(16, 5), sharey=True)

for ax, (grp, grp_color) in zip(axes, GROUP_SPECS):
    violin_data, violin_colors, xtick_labels, linestyles = [], [], [], []
    for cls, ls in COMP_CLASSES:
        d = (comp_events[(comp_events['source_group'] == grp) &
                         (comp_events['complementarity'] == cls)]['delta'].dropna())
        if len(d) < 3: continue
        violin_data.append(d.values)
        # Tint the group color: complementary=full, neutral=medium, disruptive=light
        alpha_map = {'complementary': 0.85, 'neutral': 0.5, 'disruptive': 0.25}
        violin_colors.append(grp_color)
        linestyles.append(ls)
        t, p = stats.ttest_1samp(d, popmean=0)
        xtick_labels.append(f'{cls}\nn={len(d)}\np={p:.3f}')

    if not violin_data:
        ax.set_title(grp); ax.axis('off'); continue

    parts = ax.violinplot(violin_data, positions=range(len(violin_data)),
                          showmedians=True, showextrema=False)
    alpha_vals = {'complementary': 0.8, 'neutral': 0.5, 'disruptive': 0.25}
    for pc, col, (cls, _) in zip(parts['bodies'], violin_colors, COMP_CLASSES):
        pc.set_facecolor(col)
        pc.set_alpha(alpha_vals.get(cls, 0.5))
    parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
    for i, d in enumerate(violin_data):
        ax.scatter([i]*len(d), d, color='k', alpha=0.2, s=5, zorder=3)

    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xticks(range(len(xtick_labels)))
    ax.set_xticklabels(xtick_labels, fontsize=7.5, rotation=15, ha='right')
    ax.set_title(grp, color=grp_color, fontweight='bold')
    if ax == axes[0]:
        ax.set_ylabel('Δ log2(swap/WT): carrier − non-carrier')

plt.suptitle('Carrier RAAS delta faceted by source group and swap-missense complementarity\n'
             '(darker = complementary, lighter = disruptive)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'contact_saap_complementarity_faceted.pdf'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── EVE DOUBLE-MUTANT FAVORABILITY vs RAAS ───────────────────────────────────
# For each (missense_i, swap_j) contact pair, look up the EVE double-mutant
# prediction and test whether carrier RAAS delta tracks predicted favorability.
#
# EVE files (long format): destab_{ENSP}_{ACC}_b{beta}_double_mutant_matrix.csv
#   columns: i, aa_i, j, aa_j, i_score, j_score, dmm_score
#   each row = double mutant (pos i -> aa_i) + (pos j -> aa_j); aa_* = mutant AA.
#   i_score/j_score = single-mutant EVE scores; dmm_score = double-mutant score.
#   More negative = more deleterious.
#
# Two quantities tested vs RAAS delta:
#   dmm_score : raw double-mutant favorability (higher = more tolerated)
#   epistasis : dmm_score - (i_score + j_score)
#               positive => swap makes the pair MORE tolerable than the two
#               singles independently (compensation); orientation-invariant.
#
# Self-contained: builds its own per-event delta table from df_miss (cell 14).

EVE_DIR  = '/projects/marubi/collabs/slavov_rna/aa_mut_preds/eve_alns_double_muts/destab'
EVE_BETA = '0.1'   # available betas: 0.1, 0.3, 0.5

# ── Map UniProt ACC -> EVE file path for the chosen beta ──────────────────────
eve_files = {}
for fp in glob.glob(os.path.join(EVE_DIR, f'destab_*_b{EVE_BETA}_double_mutant_matrix.csv')):
    m = _re.match(r'destab_(ENSP\d+)_([A-Z0-9]+)_b' + _re.escape(EVE_BETA)
                  + r'_double_mutant_matrix\.csv$', os.path.basename(fp))
    if m:
        eve_files[m.group(2)] = fp
print(f'EVE matrices available (beta={EVE_BETA}): {len(eve_files)} accessions')

# ── Build per-event delta table independently (no dependency on cell 15) ─────
if 'miss_aa' not in dir():
    miss_aa = {}
    for _, _r in all_miss.drop_duplicates(['SYMBOL','pos','Amino_acids']).iterrows():
        _aa = str(_r.get('Amino_acids', ''))
        if '/' in _aa:
            _w, _a = _aa.split('/')[0].strip(), _aa.split('/')[1].strip()
            if len(_w) == 1 and len(_a) == 1:
                miss_aa[(str(_r['SYMBOL']), int(_r['pos']))] = (_w, _a)

def _eve_delta(g):
    c  = g.loc[ g['is_carrier'], 'log2_raas']
    nc = g.loc[~g['is_carrier'], 'log2_raas']
    if len(c) == 0 or len(nc) == 0:
        return pd.Series({'delta': float('nan')})
    return pd.Series({'delta': c.mean() - nc.mean()})

eve_events = (df_miss[df_miss['log2_raas'].notna()]
    .groupby(['gene','miss_pos','contact_pos','swap','source_group'])
    .apply(_eve_delta)
    .reset_index()
    .dropna(subset=['delta']))
print(f'Per-event delta rows: {len(eve_events):,}')

need_accs = {gene_to_acc.get(g) for g in eve_events['gene'].unique()} & set(eve_files)
print(f'Accessions overlapping our detected events: {len(need_accs)}')

# ── Long-format lookup: (pos_i, aa_i, pos_j, aa_j) -> (dmm_score, i+j single) ─
_eve_cache = {}
def _get_eve_lut(acc):
    if acc in _eve_cache:
        return _eve_cache[acc]
    fp = eve_files.get(acc)
    if fp is None:
        _eve_cache[acc] = None; return None
    d = pd.read_csv(fp, usecols=['i','aa_i','j','aa_j','i_score','j_score','dmm_score'])
    lut = {}
    for i, ai, j, aj, si, sj, dm in zip(d['i'], d['aa_i'], d['j'], d['aa_j'],
                                        d['i_score'], d['j_score'], d['dmm_score']):
        lut[(int(i), str(ai), int(j), str(aj))] = (float(dm), float(si) + float(sj))
    _eve_cache[acc] = lut
    return lut

def eve_lookup(acc, pos_i, alt_i, pos_j, alt_j):
    """Return (dmm_score, epistasis) for the double mutant, trying both orders."""
    lut = _get_eve_lut(acc)
    if not lut:
        return (np.nan, np.nan)
    for key in ((pos_i, alt_i, pos_j, alt_j), (pos_j, alt_j, pos_i, alt_i)):
        if key in lut:
            dm, ssum = lut[key]
            return (dm, dm - ssum)
    return (np.nan, np.nan)

# ── Resolve EVE scores for each unique (gene, miss_pos, contact_pos, swap) ─────
combo = eve_events[['gene','miss_pos','contact_pos','swap']].drop_duplicates()
eve_rows = []
for _, r in combo.iterrows():
    g  = r['gene']
    mp = int(r['miss_pos']); cp = int(r['contact_pos']); sw = str(r['swap'])
    acc = gene_to_acc.get(g)
    wt_i, alt_i = miss_aa.get((g, mp), (None, None))
    mm = _re.match(r'^([A-Z])(\d+)([A-Z])$', sw)
    dm = ep = np.nan
    if acc is not None and wt_i is not None and mm:
        alt_j = mm.group(3)
        dm, ep = eve_lookup(acc, mp, alt_i, cp, alt_j)
    eve_rows.append((g, mp, cp, sw, dm, ep))
eve_df = pd.DataFrame(eve_rows,
    columns=['gene','miss_pos','contact_pos','swap','eve_dmm','eve_epist'])
n_res = eve_df['eve_dmm'].notna().sum()
print(f'\nEVE scores resolved: {n_res:,} / {len(eve_df):,} unique combos')
if n_res:
    print(f"  dmm_score  min/median/max: {eve_df['eve_dmm'].min():.3f} / "
          f"{eve_df['eve_dmm'].median():.3f} / {eve_df['eve_dmm'].max():.3f}")
    print(f"  epistasis  min/median/max: {eve_df['eve_epist'].min():.3f} / "
          f"{eve_df['eve_epist'].median():.3f} / {eve_df['eve_epist'].max():.3f}")

# ── Merge onto events and correlate vs RAAS delta ────────────────────────────
ce  = eve_events.merge(eve_df, on=['gene','miss_pos','contact_pos','swap'], how='left')
sub = ce[ce['eve_dmm'].notna() & ce['delta'].notna()].copy()
print(f'\nEvents with EVE score and delta: {len(sub):,}')

DESTAB = ['Both AM+SPURS','SPURS only','AM only']
for metric in ['eve_dmm', 'eve_epist']:
    print(f'\n── Spearman: {metric}  vs  carrier RAAS delta ───────────────────')
    for lbl, mask in [('All',     sub),
                      ('Destab',  sub[sub['source_group'].isin(DESTAB)]),
                      ('Neutral', sub[sub['source_group'] == 'Neutral'])]:
        if len(mask) < 5:
            print(f'  {lbl:<8} n={len(mask)} (too few)'); continue
        rho, p = stats.spearmanr(mask[metric], mask['delta'])
        print(f'  {lbl:<8} n={len(mask):>4}  spearman rho={rho:+.3f}  p={p:.3g}')

# ── Terciles of epistasis (the compensation axis) -> median RAAS delta ───────
dsub = sub[sub['source_group'].isin(DESTAB)].copy()
if len(dsub) >= 9:
    dsub['epist_tercile'] = pd.qcut(dsub['eve_epist'], 3,
                                    labels=['low','mid','high'], duplicates='drop')
    print('\n── Median RAAS delta by epistasis tercile (destab; high=compensatory) ─')
    print(dsub.groupby('epist_tercile')['delta'].agg(['size','median','mean']).to_string())

# ── Plots: dmm scatter | epistasis scatter | epistasis-tercile violin ────────
if len(sub) >= 5:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    grp_colors = [('Both AM+SPURS','#d62728'),('SPURS only','#ff7f0e'),
                  ('AM only','#9467bd'),('Neutral','#2ca02c')]
    for ax, metric, xlab in [
            (axes[0], 'eve_dmm',  'EVE double-mutant score (higher = tolerated)'),
            (axes[1], 'eve_epist','EVE epistasis = dmm - (i+j)  (higher = compensatory)')]:
        for grp, col in grp_colors:
            gd = sub[sub['source_group'] == grp]
            if len(gd):
                ax.scatter(gd[metric], gd['delta'], s=18, alpha=0.6,
                           color=col, label=f'{grp} (n={len(gd)})')
        ax.axhline(0, color='k', lw=0.8, ls='--')
        ax.set_xlabel(xlab)
        ax.set_ylabel('Δ log2(swap/WT): carrier − non-carrier')
    axes[0].set_title('RAAS delta vs raw double-mutant favorability')
    axes[1].set_title('RAAS delta vs epistasis (compensation)')
    axes[1].legend(fontsize=8)

    if 'epist_tercile' in dsub.columns:
        vdata, vlabels = [], []
        for t in ['low','mid','high']:
            d = dsub[dsub['epist_tercile'] == t]['delta'].dropna()
            if len(d) >= 3:
                vdata.append(d.values); vlabels.append(f'{t}\nn={len(d)}')
        if vdata:
            parts = axes[2].violinplot(vdata, positions=range(len(vdata)),
                                       showmedians=True, showextrema=False)
            for pc in parts['bodies']:
                pc.set_facecolor('#1f77b4'); pc.set_alpha(0.6)
            parts['cmedians'].set_color('black'); parts['cmedians'].set_linewidth(2)
            for i, d in enumerate(vdata):
                axes[2].scatter([i]*len(d), d, color='k', alpha=0.2, s=6)
            axes[2].axhline(0, color='k', lw=0.8, ls='--')
            axes[2].set_xticks(range(len(vlabels)))
            axes[2].set_xticklabels(vlabels)
            axes[2].set_ylabel('Δ log2(swap/WT): carrier − non-carrier')
            axes[2].set_title('RAAS delta by epistasis tercile (destab)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'contact_saap_eve_double.pdf'),
                dpi=150, bbox_inches='tight')
    plt.show()

# ── Top compensatory (high-epistasis) detected double mutants ────────────────
if len(sub):
    print('\n── Most compensatory (highest epistasis) detected pairs (destab) ─────')
    top = (sub[sub['source_group'].isin(DESTAB)]
           .sort_values('eve_epist', ascending=False)
           [['gene','miss_pos','contact_pos','swap','source_group',
             'eve_dmm','eve_epist','delta']].head(20))
    print(top.to_string(index=False))

ce.to_csv(os.path.join(OUT_DIR, 'contact_saap_eve_double.tsv'), sep='\t', index=False)
print(f'\nSaved -> {OUT_DIR}/contact_saap_eve_double.tsv')


In [ ]:
# ── SAVE ──────────────────────────────────────────────────────────────────────
df.to_csv(os.path.join(OUT_DIR, 'contact_saap_records.tsv'), sep='\t', index=False)
print(f'Saved to {OUT_DIR}')